# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [5]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
#if importlib.util.find_spec("pysqlite3") is not None:
#    import pysqlite3
#    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

__import__('pysqlite3')
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [6]:
# TODO: Import the necessary libs 
import os
from dotenv import load_dotenv
import chromadb
from tavily import TavilyClient

from pydantic import BaseModel

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [7]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY=os.getenv("CHROMA_OPENAI_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
BASE_URL=os.getenv("BASE_URL")

llm = LLM(
    model="gpt-4o-mini",
    base_url=BASE_URL,
    api_key=OPENAI_API_KEY
)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [8]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

chroma_client = chromadb.PersistentClient(path="/workspace/Code/project/starter")
collection = chroma_client.get_collection("udaplay")

@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=5
    )

    documents = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        documents.append({
            "Platform": metadata.get("Platform", ""),
            "Name": metadata.get("Name", ""),
            "YearOfRelease": metadata.get("YearOfRelease", ""),
            "Description": doc
        })

    return documents

Failed to send telemetry event ClientStartEvent: module 'chromadb' has no attribute 'get_settings'


#### Evaluate Retrieval Tool

In [9]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationReport(BaseModel):
    useful: bool
    description: str

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """

    docs_text = "\n".join([str(doc) for doc in retrieved_docs])

    messages = [
        SystemMessage(content=(
            "Your task is to evaluate if the documents are enough to respond the query. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not."
        )),
        UserMessage(content=(
            f"Query: {question}\n\n"
            f"Retrieved Documents:\n{docs_text}"
        ))
    ]

    response = llm.invoke(
        input=messages,
        response_format=EvaluationReport
    )

    report = EvaluationReport.model_validate_json(response.content)

    return {"useful": report.useful, "description": report.description}

#### Game Web Search Tool

In [10]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question: str) -> str:
    """
    Web search: Searches the internet for game industry information
    when the vector DB results are not sufficient.
    args:
    - question: a question about game industry.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    results = client.search(query=question)

    return results["results"]

### Agent

In [11]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

agent = Agent(
    api_key=OPENAI_API_KEY,
    model_name="gpt-4o-mini",
    base_url=BASE_URL,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions=(
        "You are UdaPlay, an AI Research Agent specialized in the video game industry. "
        "Follow this workflow:\n"
        "1. When the user asks a question, first use retrieve_game to search the vector DB.\n"
        "2. Then use evaluate_retrieval to assess if the results are sufficient.\n"
        "3. If the evaluation says the documents are NOT useful, use game_web_search as a fallback.\n"
        "4. Always provide a clear, structured answer based on the best available information.\n"
        "5. Cite your source (vector DB or web search) in your response."
    )
)

In [14]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    run = agent.invoke(question)
    final_state = run.get_final_state()
    last_ai_message = final_state["messages"][-1]
    print(f"A: {last_ai_message.content}")


Q: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
A: Pokémon Gold and Silver were released in 1999 for the Game Boy Color. They are part of the second generation of Pokémon games, introducing new Pokémon, regions, and gameplay mechanics.

(Source: Vector DB)

Q: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
A: The first 3D platformer Mario game is **Super Mario 64**, which was released in 1996 for the Nintendo 64. This game is considered groundbreaking for its time, setting new standards for the 3D platformer genre and featuring Mario's quest to rescue Princess Peach.

(Source: Vector DB)

Q: Was Mortal Kombat X released for Playstation 5?
[StateMachine] 

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes